In [1]:
import pandas as pd
import numpy as np
import random
import copy
import math
from datetime import date
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

# in some Compeletion Report LSNAMECODE is splited in some it is not so have to check that
def edit_ls_code_column(x):
    value=x.split('_')
    if len(value)>3:
        route_value="_".join(value[:-1])
    else:
        route_value="_".join(value)
    return route_value

# for generated file version
version=2
project_name='UTA'
today_date = date.today()
today_date=''.join(str(today_date).split('-'))


detail_df_stops=pd.read_excel('details_project_od_excel_UTA.xlsx',sheet_name='STOPS')
detail_df_xfers=pd.read_excel('details_project_od_excel_UTA.xlsx',sheet_name='XFERS')

In [2]:

wkend_overall_df=pd.read_excel('UTA_SL_CR.xlsx',sheet_name='WkEND-RAIL')
# wkend_overall_df['LS_NAME_CODE']=wkend_overall_df['LS_NAME_CODE'].apply(edit_ls_code_column)
wkend_route_df=pd.read_excel('UTA_SL_CR.xlsx',sheet_name='WkEND-RailTotal')

df=pd.read_csv("elvisutaobweekday2024_export_odbc.csv")

wkday_overall_df=pd.read_excel('UTA_SL_CR.xlsx',sheet_name='WkDAY-RAIL')
# wkday_overall_df['LS_NAME_CODE']=wkday_overall_df['LS_NAME_CODE'].apply(edit_ls_code_column)
wkday_route_df=pd.read_excel('UTA_SL_CR.xlsx',sheet_name='WkDAY-RailTotal')

df['ROUTE_SURVEYEDCode_Splited']=df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(str(x).split('_')[:-1]) )
stop_on_clntid=['stoponclntid']
stop_on_clntid=check_all_characters_present(df,stop_on_clntid)
df['STATION_ID_SPLITTED']=df[stop_on_clntid[0]].apply(lambda x:str(x).split('_')[-1])

In [3]:
wkend_overall_df['STATION_ID_SPLITTED']=wkend_overall_df['STATION_ID'].apply(lambda x: str(x).split('_')[-1])
wkday_overall_df['STATION_ID_SPLITTED']=wkday_overall_df['STATION_ID'].apply(lambda x: str(x).split('_')[-1])

In [4]:
wkday_route_df['ROUTE_TOTAL'] = pd.to_numeric(wkday_route_df['ROUTE_TOTAL'], errors='coerce')
wkday_route_df['ROUTE_TOTAL'].fillna(0, inplace=True)
wkend_route_df['ROUTE_TOTAL'] = pd.to_numeric(wkend_route_df['ROUTE_TOTAL'], errors='coerce')
wkend_route_df['ROUTE_TOTAL'].fillna(0, inplace=True)

wkday_route_df['ROUTE_TOTAL'] = np.ceil(wkday_route_df['ROUTE_TOTAL']).astype(int)
wkend_route_df['ROUTE_TOTAL'] = np.ceil(wkend_route_df['ROUTE_TOTAL']).astype(int)

wkday_overall_df[[0,1,2,3,4,5]]=wkday_overall_df[[0,1,2,3,4,5]].fillna(0)
wkend_overall_df[[0,1,2,3,4,5]]=wkend_overall_df[[0,1,2,3,4,5]].fillna(0)

In [5]:

time_column_check=['timeoncode']
time_period_column_check=['timeon']
time_column=check_all_characters_present(df,time_column_check)
time_period_column=check_all_characters_present(df,time_period_column_check)
route_survey_column_check=['routesurveyedcode']
route_survey_column=check_all_characters_present(df,route_survey_column_check)
stopon_clntid_column_check=['stoponclntid']
stopon_clntid_column=check_all_characters_present(df,stopon_clntid_column_check)
stopoff_clntid_column_check=['stopoffclntid']
stopoff_clntid_column=check_all_characters_present(df,stopoff_clntid_column_check)


df.drop_duplicates(subset='id',inplace=True)
df=df[df['INTERV_INIT']!='999']
df=df[df['INTERV_INIT']!=999]



date_columns_check=['completed','datestarted']
date_columns=check_all_characters_present(df,date_columns_check)

def determine_date(row):
    if not pd.isnull(row[date_columns[0]]):
        return row[date_columns[0]]
    elif not pd.isnull(row[date_columns[1]]):
        return row[date_columns[1]]
    else:
        return pd.NaT

df['Date'] = df.apply(determine_date, axis=1)


In [6]:
def get_day_name(x):
    if pd.isna(x):
        return None  # or another appropriate default value
    formats_to_check = ['%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M']
    
    for format_str in formats_to_check:
        try:
            date_object = datetime.strptime(x, format_str)
            day_name = date_object.strftime('%A')
            return day_name
        except ValueError:
            continue

df['Day']=df['Date'].apply(get_day_name)

df['LAST_SURVEY_DATE'] = pd.to_datetime(df['Date'], errors='coerce')
latest_date = df['LAST_SURVEY_DATE'].max()
latest_date_df = pd.DataFrame({'Latest_Survey_Date': [latest_date]})

weekend_df=df[df['Day'].isin(['Saturday','Sunday'])]

weekday_df=df[~(df['Day'].isin(['Saturday','Sunday']))]
weekend_df.dropna(subset=[time_column[0]],inplace=True)
weekday_df.dropna(subset=[time_column[0]],inplace=True)

In [7]:
wkend_overall_df.dropna(subset=['LS_NAME_CODE'],inplace=True)
wkday_overall_df.dropna(subset=['LS_NAME_CODE'],inplace=True)

In [17]:
weekend_df.to_csv('UTA Weekend Data.csv',index=False)

In [8]:

def create_time_value_df_with_display(overall_df, df):
    """
    Create a time-value DataFrame summarizing counts and time ranges, considering only records where 
    'ROUTE_SURVEYEDCode' from overall_df matches 'ROUTE_SURVEYEDCode_Splited' in df.

    Parameters:
        overall_df (pd.DataFrame): DataFrame with the column 'ROUTE_SURVEYEDCode'.
        df (pd.DataFrame): Input DataFrame containing the time values and 'ROUTE_SURVEYEDCode_Splited'.
        time_column (str): Name of the column in the input DataFrame containing the time values.

    Returns:
        pd.DataFrame: Processed DataFrame with counts, time ranges, and display text.
    """
    # Filter df where overall_df['ROUTE_SURVEYEDCode'] == df['ROUTE_SURVEYEDCode_Splited']
    matched_df = df[df['ROUTE_SURVEYEDCode_Splited'].isin(overall_df['ROUTE_SURVEYEDCode'])]

    # Define time value groups
    pre_early_am_values = [1]
    early_am_values = [2]
    am_values = [3, 4, 5, 6]
    midday_values = [7, 8, 9, 10, 11]
    pm_values = [12, 13, 14]
    evening_values = [15, 16, 17, 18]

    # Mapping time groups to corresponding columns
    time_group_mapping = {
        0: pre_early_am_values,
        1: early_am_values,
        2: am_values,
        3: midday_values,
        4: pm_values,
        5: evening_values,
    }

    # Mapping time values to time ranges
    time_mapping = {
        1: 'Before 5:00 am',
        2: '5:00 am - 6:00 am',
        3: '6:00 am - 7:00 am',
        4: '7:00 am - 8:00 am',
        5: '8:00 am - 9:00 am',
        6: '9:00 am - 10:00 am',
        7: '10:00 am - 11:00 am',
        8: '11:00 am - 12:00 pm',
        9: '12:00 pm - 1:00 pm',
        10: '1:00 pm - 2:00 pm',
        11: '2:00 pm - 3:00 pm',
        12: '3:00 pm - 4:00 pm',
        13: '4:00 pm - 5:00 pm',
        14: '5:00 pm - 6:00 pm',
        15: '6:00 pm - 7:00 pm',
        16: '7:00 pm - 8:00 pm',
        17: '8:00 pm - 9:00 pm',
        18: 'After 9:00 pm',
    }

    # Ensure the time_column is of integer type
    matched_df[time_column] = matched_df[time_column].fillna(0).astype(int)

    # Initialize the new DataFrame
    new_df = pd.DataFrame(columns=["Original Text", 0, 1, 2, 3, 4, 5])

    # Populate the DataFrame with counts
    for col, values in time_group_mapping.items():
        for value in values:
            count = matched_df[matched_df[time_column[0]] == value].shape[0]
            row = {"Original Text": value}

            # Initialize all columns to 0
            for c in range(6):
                row[c] = 0

            # Update the corresponding column with the count
            row[col] = count
            new_df = pd.concat([new_df, pd.DataFrame([row])], ignore_index=True)

    # Map time values to time ranges
    new_df['Time Range'] = new_df['Original Text'].map(time_mapping)

    # Drop rows with missing time ranges
    new_df.dropna(subset=['Time Range'], inplace=True)

    # Add a display text column with sequential numbering
    new_df['Display_Text'] = range(1, len(new_df) + 1)

    return new_df


In [37]:

# To create Route_SurveyedCode Direction wise comparison in terms of time values
def create_route_direction_level_df(overalldf, df,time):
    pre_early_am_values = [1]
    early_am_values = [2]
    am_values = [3, 4, 5, 6]
    midday_values = [7, 8, 9, 10, 11]
    pm_values = [12, 13, 14]
    evening_values = [15, 16, 17, 18]

    pre_early_am_column = [0]  # 0 is for Pre-Early AM header
    early_am_column = [1]  # 1 is for Early AM header
    am_column = [2]  # This is for AM header
    midday_column = [3]  # this is for MIDDAY header
    pm_column = [4]  # this is for PM header
    evening_column = [5]  # this is for EVENING header

    def convert_string_to_integer(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return 0

    # Creating new dataframe for specifically AM, PM, MIDDAY, Evening data and added values from Completion Report
    new_df = pd.DataFrame()
    new_df['ROUTE_SURVEYEDCode']=overalldf['LS_NAME_CODE']
    if time=='weekend':
        new_df['Day'] = overalldf['DAY']
    
    new_df['STATION_ID']=overalldf['STATION_ID']
    new_df['STATION_ID_SPLITTED']=overalldf['STATION_ID_SPLITTED']
    new_df['CR_PRE_Early_AM'] = pd.to_numeric(overalldf[pre_early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Early_AM'] = pd.to_numeric(overalldf[early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_AM_Peak'] = pd.to_numeric(overalldf[am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Midday'] = pd.to_numeric(overalldf[midday_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_PM_Peak'] = pd.to_numeric(overalldf[pm_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Evening'] = pd.to_numeric(overalldf[evening_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    # print("new_df_columns",new_df.columns)
    new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
    new_df.fillna(0,inplace=True)
#     new code added for merging the same ROUTE_SURVEYEDCode
    # new_df=new_df.groupby('ROUTE_SURVEYEDCode', as_index=False).sum()
    # new_df.reset_index(drop=True, inplace=True)

    for index, row in new_df.iterrows():
        route_code = row['ROUTE_SURVEYEDCode']
        station_id=row['STATION_ID_SPLITTED']
        if time=='weekend':
            day=row['Day']
        def get_counts_and_ids(time_values):
            # Just for SALEM
            # subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code) & (df[time_column[0]].isin(time_values))]
            if time=='weekend':
                subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code)&(df['STATION_ID_SPLITTED']==station_id)  & (df[time_column[0]].isin(time_values))& 
                 (df['Day'].str.lower() == str(day).lower())]
            else:
                subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code)&(df['STATION_ID_SPLITTED']==station_id)  & (df[time_column[0]].isin(time_values))]
            subset_df=subset_df.drop_duplicates(subset='id')
            count = subset_df.shape[0]
            ids = subset_df['id'].values
            return count, ids

        pre_early_am_value, pre_early_am_value_ids = get_counts_and_ids(pre_early_am_values)
        early_am_value, early_am_value_ids = get_counts_and_ids(early_am_values)
        am_value, am_value_ids = get_counts_and_ids(am_values)
        midday_value, midday_value_ids = get_counts_and_ids(midday_values)
        pm_value, pm_value_ids = get_counts_and_ids(pm_values)
        evening_value, evening_value_ids = get_counts_and_ids(evening_values)
        
        new_df.loc[index, 'CR_Total'] = row['CR_PRE_Early_AM']+row['CR_Early_AM']+row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
        new_df.loc[index, 'CR_AM_Peak'] =row['CR_AM_Peak']
        # new_df.loc[index, 'CR_AM_Peak'] =row['CR_PRE_EARLY_AM']+row['CR_EARLY_AM']+ row['CR_AM_Peak']
        new_df.loc[index, 'DB_PRE_Early_AM_Peak'] = pre_early_am_value
        new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
        new_df.loc[index, 'DB_AM_Peak'] = am_value
        new_df.loc[index, 'DB_Midday'] = midday_value
        new_df.loc[index, 'DB_PM_Peak'] = pm_value
        new_df.loc[index, 'DB_Evening'] = evening_value
        new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value+pre_early_am_value+early_am_value
        
    #     new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(x.split('_')[:-1]) )
        route_code_level_df=pd.DataFrame()

        unique_routes=new_df['ROUTE_SURVEYEDCode'].unique()

        route_code_level_df['ROUTE_SURVEYEDCode']=unique_routes

        # weekend_df.rename(columns={'ROUTE_TOTAL':'CR_Overall_Goal','SURVEY_ROUTE_CODE':'ROUTE_SURVEYEDCode','LS_NAME_CODE':'ROUTE_SURVEYEDCode'},inplace=True)

        for index, row in new_df.iterrows():
            pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
            early_am_peak_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
            am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
            midday_diff=row['CR_Midday']-row['DB_Midday']    
            pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
            evening_diff=row['CR_Evening']-row['DB_Evening']
            total_diff=row['CR_Total']-row['DB_Total']
    #         overall_difference=row['CR_Overall_Goal']-row['DB_Total']
            new_df.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
            new_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
            new_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
            new_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
            new_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
            new_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
            # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
            new_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))

    return new_df


In [38]:
wkday_route_direction_df=create_route_direction_level_df(wkday_overall_df,weekday_df,None)
wkday_route_direction_df

,ROUTE_SURVEYEDCode,STATION_ID,STATION_ID_SPLITTED,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,...,DB_PM_Peak,DB_Evening,DB_Total,PRE_Early_AM_DIFFERENCE,Early_AM_DIFFERENCE,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
0,UTA_1_703_00,UTA_1_703_19171,19171,0.0,0.0,4.0,14.0,28.0,15.0,61.0,...,31.0,13.0,88.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0
1,UTA_1_703_00,UTA_1_703_19169,19169,0.0,0.0,2.0,10.0,12.0,14.0,38.0,...,20.0,14.0,45.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,UTA_1_703_00,UTA_1_703_19167,19167,0.0,0.0,1.0,15.0,16.0,10.0,42.0,...,15.0,10.0,47.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3,UTA_1_703_00,UTA_1_703_18384,18384,0.0,0.0,2.0,31.0,30.0,18.0,81.0,...,28.0,35.0,92.0,0.0,0.0,0.0,6.0,2.0,0.0,8.0
4,UTA_1_703_00,UTA_1_703_18381,18381,0.0,0.0,3.0,7.0,5.0,4.0,19.0,...,5.0,4.0,26.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,UTA_1_750_01,UTA_1_750_01_23074,23074,0.0,0.0,6.0,7.0,6.0,5.0,24.0,...,12.0,4.0,34.0,0.0,0.0,3.0,0.0,0.0,1.0,4.0
172,UTA_1_750_01,UTA_1_750_01_23071,23071,0.0,0.0,6.0,3.0,3.0,4.0,16.0,...,6.0,5.0,20.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0
173,UTA_1_750_01,UTA_1_750_01_25330,25330,0.0,0.0,2.0,2.0,2.0,2.0,8.0,...,1.0,1.0,5.0,0.0,0.0,0.0,1.0,1.0,1.0,3.0
174,UTA_1_750_01,UTA_1_750_01_23076,23076,0.0,0.0,3.0,3.0,3.0,2.0,11.0,...,6.0,3.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [41]:
wkday_route_direction_df[wkday_route_direction_df['STATION_ID']=='UTA_1_703_19171']

,ROUTE_SURVEYEDCode,STATION_ID,STATION_ID_SPLITTED,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,...,DB_PM_Peak,DB_Evening,DB_Total,PRE_Early_AM_DIFFERENCE,Early_AM_DIFFERENCE,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
0,UTA_1_703_00,UTA_1_703_19171,19171,0.0,0.0,4.0,14.0,28.0,15.0,61.0,...,31.0,13.0,88.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0


In [13]:

# To create Route_SurveyedCode Direction wise comparison in terms of time values
def create_wkend_route_direction_level_df(overalldf, df):
    pre_early_am_values = [1]
    early_am_values = [2]
    am_values = [3, 4, 5, 6]
    midday_values = [7, 8, 9, 10, 11]
    pm_values = [12, 13, 14]
    evening_values = [15, 16, 17, 18]

    pre_early_am_column = [0]  # 0 is for Pre-Early AM header
    early_am_column = [1]  # 1 is for Early AM header
    am_column = [2]  # This is for AM header
    midday_column = [3]  # this is for MIDDAY header
    pm_column = [4]  # this is for PM header
    evening_column = [5]  # this is for EVENING header

    def convert_string_to_integer(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return 0

    # Creating new dataframe for specifically AM, PM, MIDDAY, Evening data and added values from Completion Report
    new_df = pd.DataFrame()
    new_df['ROUTE_SURVEYEDCode']=overalldf['LS_NAME_CODE']
    new_df['Day'] = overalldf['DAY']
    new_df['STATION_ID']=overalldf['STATION_ID']
    new_df['STATION_ID_SPLITTED']=overalldf['STATION_ID_SPLITTED']
    new_df['CR_PRE_Early_AM'] = pd.to_numeric(overalldf[pre_early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Early_AM'] = pd.to_numeric(overalldf[early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_AM_Peak'] = pd.to_numeric(overalldf[am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Midday'] = pd.to_numeric(overalldf[midday_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_PM_Peak'] = pd.to_numeric(overalldf[pm_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Evening'] = pd.to_numeric(overalldf[evening_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    # print("new_df_columns",new_df.columns)
    new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
    new_df.fillna(0,inplace=True)
#     new code added for merging the same ROUTE_SURVEYEDCode
    # new_df=new_df.groupby('ROUTE_SURVEYEDCode', as_index=False).sum()
    # new_df.reset_index(drop=True, inplace=True)

    for index, row in new_df.iterrows():
        route_code = row['ROUTE_SURVEYEDCode']
        station_id=row['STATION_ID_SPLITTED']
        day = row['Day']

        def get_counts_and_ids(time_values):
            # Just for SALEM
            # subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code) & (df[time_column[0]].isin(time_values))]
            subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code)&(df['STATION_ID_SPLITTED']==station_id)  & (df[time_column[0]].isin(time_values))& 
                         (df['Day'].str.lower() == str(day).lower())]
            subset_df=subset_df.drop_duplicates(subset='id')
            count = subset_df.shape[0]
            ids = subset_df['id'].values
            return count, ids

        pre_early_am_value, pre_early_am_value_ids = get_counts_and_ids(pre_early_am_values)
        early_am_value, early_am_value_ids = get_counts_and_ids(early_am_values)
        am_value, am_value_ids = get_counts_and_ids(am_values)
        midday_value, midday_value_ids = get_counts_and_ids(midday_values)
        pm_value, pm_value_ids = get_counts_and_ids(pm_values)
        evening_value, evening_value_ids = get_counts_and_ids(evening_values)
        
        new_df.loc[index, 'CR_Total'] = row['CR_PRE_Early_AM']+row['CR_Early_AM']+row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
        new_df.loc[index, 'CR_AM_Peak'] =row['CR_AM_Peak']
        # new_df.loc[index, 'CR_AM_Peak'] =row['CR_PRE_EARLY_AM']+row['CR_EARLY_AM']+ row['CR_AM_Peak']
        new_df.loc[index, 'DB_PRE_Early_AM_Peak'] = pre_early_am_value
        new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
        new_df.loc[index, 'DB_AM_Peak'] = am_value
        new_df.loc[index, 'DB_Midday'] = midday_value
        new_df.loc[index, 'DB_PM_Peak'] = pm_value
        new_df.loc[index, 'DB_Evening'] = evening_value
        new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value+pre_early_am_value+early_am_value
        
    #     new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(x.split('_')[:-1]) )
        route_code_level_df=pd.DataFrame()

        unique_routes=new_df['ROUTE_SURVEYEDCode'].unique()

        route_code_level_df['ROUTE_SURVEYEDCode']=unique_routes

        # weekend_df.rename(columns={'ROUTE_TOTAL':'CR_Overall_Goal','SURVEY_ROUTE_CODE':'ROUTE_SURVEYEDCode','LS_NAME_CODE':'ROUTE_SURVEYEDCode'},inplace=True)

        for index, row in new_df.iterrows():
            pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
            early_am_peak_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
            am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
            midday_diff=row['CR_Midday']-row['DB_Midday']    
            pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
            evening_diff=row['CR_Evening']-row['DB_Evening']
            total_diff=row['CR_Total']-row['DB_Total']
    #         overall_difference=row['CR_Overall_Goal']-row['DB_Total']
            new_df.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
            new_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
            new_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
            new_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
            new_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
            new_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
            # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
            new_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))

    return new_df


In [ ]:
def create_uta_route_direction_level_df(overalldf, df,time):
    pre_early_am_values = [1]
    early_am_values = [2]
    am_values = [3, 4, 5, 6]
    midday_values = [7, 8, 9, 10, 11]
    pm_values = [12, 13, 14]
    evening_values = [15, 16, 17, 18]

    pre_early_am_column = [0]  # 0 is for Pre-Early AM header
    early_am_column = [1]  # 1 is for Early AM header
    am_column = [2]  # This is for AM header
    midday_column = [3]  # this is for MIDDAY header
    pm_column = [4]  # this is for PM header
    evening_column = [5]  # this is for EVENING header

    def convert_string_to_integer(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return 0

    # Creating new dataframe for specifically AM, PM, MIDDAY, Evening data and added values from Completion Report
    new_df = pd.DataFrame()
    new_df['ROUTE_SURVEYEDCode']=overalldf['LS_NAME_CODE']
    if time=='weekend':
        new_df['Day'] = overalldf['DAY']
    
    new_df['STATION_ID']=overalldf['STATION_ID']
    new_df['STATION_ID_SPLITTED']=overalldf['STATION_ID_SPLITTED']
    new_df['CR_PRE_Early_AM'] = pd.to_numeric(overalldf[pre_early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Early_AM'] = pd.to_numeric(overalldf[early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_AM_Peak'] = pd.to_numeric(overalldf[am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Midday'] = pd.to_numeric(overalldf[midday_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_PM_Peak'] = pd.to_numeric(overalldf[pm_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Evening'] = pd.to_numeric(overalldf[evening_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    # print("new_df_columns",new_df.columns)
    new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
    new_df.fillna(0,inplace=True)
#     new code added for merging the same ROUTE_SURVEYEDCode
    # new_df=new_df.groupby('ROUTE_SURVEYEDCode', as_index=False).sum()
    # new_df.reset_index(drop=True, inplace=True)

    for index, row in new_df.iterrows():
        route_code = row['ROUTE_SURVEYEDCode']
        station_id=row['STATION_ID_SPLITTED']
        if time=='weekend':
            day=row['Day']
        def get_counts_and_ids(time_values):
            # Just for SALEM
            # subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code) & (df[time_column[0]].isin(time_values))]
            if time=='weekend':
                subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code)&(df['STATION_ID_SPLITTED']==station_id)  & (df[time_column[0]].isin(time_values))& 
                 (df['Day'].str.lower() == str(day).lower())]
            else:
                subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code)&(df['STATION_ID_SPLITTED']==station_id)  & (df[time_column[0]].isin(time_values))]
            subset_df=subset_df.drop_duplicates(subset='id')
            count = subset_df.shape[0]
            ids = subset_df['id'].values
            return count, ids

        pre_early_am_value, pre_early_am_value_ids = get_counts_and_ids(pre_early_am_values)
        early_am_value, early_am_value_ids = get_counts_and_ids(early_am_values)
        am_value, am_value_ids = get_counts_and_ids(am_values)
        midday_value, midday_value_ids = get_counts_and_ids(midday_values)
        pm_value, pm_value_ids = get_counts_and_ids(pm_values)
        evening_value, evening_value_ids = get_counts_and_ids(evening_values)
        
        new_df.loc[index, 'CR_Total'] = row['CR_PRE_Early_AM']+row['CR_Early_AM']+row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
        new_df.loc[index, 'CR_AM_Peak'] =row['CR_AM_Peak']
        # new_df.loc[index, 'CR_AM_Peak'] =row['CR_PRE_EARLY_AM']+row['CR_EARLY_AM']+ row['CR_AM_Peak']
        new_df.loc[index, 'DB_PRE_Early_AM_Peak'] = pre_early_am_value
        new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
        new_df.loc[index, 'DB_AM_Peak'] = am_value
        new_df.loc[index, 'DB_Midday'] = midday_value
        new_df.loc[index, 'DB_PM_Peak'] = pm_value
        new_df.loc[index, 'DB_Evening'] = evening_value
        new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value+pre_early_am_value+early_am_value
        
    #     new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(x.split('_')[:-1]) )
        route_code_level_df=pd.DataFrame()

        unique_routes=new_df['ROUTE_SURVEYEDCode'].unique()

        route_code_level_df['ROUTE_SURVEYEDCode']=unique_routes

        # weekend_df.rename(columns={'ROUTE_TOTAL':'CR_Overall_Goal','SURVEY_ROUTE_CODE':'ROUTE_SURVEYEDCode','LS_NAME_CODE':'ROUTE_SURVEYEDCode'},inplace=True)

        for index, row in new_df.iterrows():
            pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
            early_am_peak_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
            am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
            midday_diff=row['CR_Midday']-row['DB_Midday']    
            pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
            evening_diff=row['CR_Evening']-row['DB_Evening']
            total_diff=row['CR_Total']-row['DB_Total']
    #         overall_difference=row['CR_Overall_Goal']-row['DB_Total']
            new_df.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
            new_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
            new_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
            new_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
            new_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
            new_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
            # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
            new_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))

    return new_df


In [39]:
wkend_route_direction_df=create_route_direction_level_df(wkend_overall_df,weekend_df,'weekend')
# wkend_route_direction_df=create_wkend_route_direction_level_df(wkend_overall_df,weekend_df)
wkend_route_direction_df

,ROUTE_SURVEYEDCode,Day,STATION_ID,STATION_ID_SPLITTED,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,...,DB_PM_Peak,DB_Evening,DB_Total,PRE_Early_AM_DIFFERENCE,Early_AM_DIFFERENCE,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
0,UTA_1_703_00,SATURDAY,UTA_1_703_19171,19171,0.0,0.0,1.0,1.0,1.0,1.0,...,3.0,1.0,14.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,UTA_1_703_00,SATURDAY,UTA_1_703_19169,19169,0.0,0.0,1.0,1.0,1.0,1.0,...,0.0,0.0,4.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0
2,UTA_1_703_00,SATURDAY,UTA_1_703_19167,19167,0.0,0.0,1.0,1.0,1.0,1.0,...,3.0,0.0,6.0,0.0,0.0,1.0,0.0,0.0,1.0,2.0
3,UTA_1_703_00,SATURDAY,UTA_1_703_18384,18384,0.0,0.0,1.0,1.0,1.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
4,UTA_1_703_00,SATURDAY,UTA_1_703_18381,18381,0.0,0.0,1.0,1.0,1.0,1.0,...,0.0,0.0,3.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,UTA_1_750_01,SUNDAY,UTA_1_750_01_23074,23074,0.0,0.0,2.0,2.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,2.0,2.0,2.0,2.0,8.0
348,UTA_1_750_01,SUNDAY,UTA_1_750_01_23071,23071,0.0,0.0,2.0,2.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,2.0,2.0,2.0,2.0,8.0
349,UTA_1_750_01,SUNDAY,UTA_1_750_01_25330,25330,0.0,0.0,2.0,2.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,2.0,2.0,2.0,2.0,8.0
350,UTA_1_750_01,SUNDAY,UTA_1_750_01_23076,23076,0.0,0.0,2.0,2.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,2.0,2.0,2.0,2.0,8.0


In [40]:
wkend_route_direction_df[wkend_route_direction_df['STATION_ID']=='UTA_1_703_19171']

,ROUTE_SURVEYEDCode,Day,STATION_ID,STATION_ID_SPLITTED,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,...,DB_PM_Peak,DB_Evening,DB_Total,PRE_Early_AM_DIFFERENCE,Early_AM_DIFFERENCE,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
0,UTA_1_703_00,SATURDAY,UTA_1_703_19171,19171,0.0,0.0,1.0,1.0,1.0,1.0,...,3.0,1.0,14.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
176,UTA_1_703_00,SUNDAY,UTA_1_703_19171,19171,0.0,0.0,2.0,2.0,2.0,2.0,...,3.0,6.0,9.0,0.0,0.0,2.0,2.0,0.0,0.0,4.0


In [16]:
# wkend_route_direction_df.to_csv("UTA_WKEND ROUTE DIRECTION.csv",index=False)

In [32]:
def create_uta_station_wise_route_level_df(overall_df,df,time):
    pre_early_am_values = [1]
    early_am_values = [2]
    am_values = [3, 4, 5, 6]
    midday_values = [7, 8, 9, 10, 11]
    pm_values = [12, 13, 14]
    evening_values = [15, 16, 17, 18]

    pre_early_am_column=[0]  #0 is for Pre-Early AM header
    early_am_column=[1]  #1 is for Early AM header
    am_column=[2] #This is for AM header
    midday_column=[3] #this is for MIDDAY header
    pm_column=[4] #this is for PM header
    evening_column=[5] #this is for EVENING header

    def convert_string_to_integer(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return 0

    # Creating new dataframe for specifically AM, PM, MIDDAY, Evenving data and added values from Compeletion Report
    new_df=pd.DataFrame()
    new_df['ROUTE_SURVEYEDCode']=overall_df['LS_NAME_CODE']
    if time=='weekend':
        new_df['Day'] = overall_df['DAY']
    new_df['STATION_ID']=overall_df['STATION_ID']
    new_df['STATION_ID_SPLITTED']=overall_df['STATION_ID_SPLITTED']
    new_df['CR_PRE_Early_AM'] = pd.to_numeric(overall_df[pre_early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Early_AM'] = pd.to_numeric(overall_df[early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_AM_Peak'] = pd.to_numeric(overall_df[am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Midday'] = pd.to_numeric(overall_df[midday_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_PM_Peak'] = pd.to_numeric(overall_df[pm_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Evening'] = pd.to_numeric(overall_df[evening_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
    new_df.fillna(0,inplace=True)
    new_df['ROUTE_SURVEYEDCode_Splitted']=new_df['ROUTE_SURVEYEDCode'].apply(edit_ls_code_column)
    
    for index, row in new_df.iterrows():
        route_code = row['ROUTE_SURVEYEDCode_Splitted']
        station_id=row['STATION_ID_SPLITTED']
        if time=='weekend':
            day=row['Day']
        def get_counts_and_ids(time_values):
            # Just for SALEM
            # subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code) & (df[time_column[0]].isin(time_values))]
            if time=='weekend':
                subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code)& (df['STATION_ID_SPLITTED']==station_id)& (df[time_column[0]].isin(time_values))& 
                         (df['Day'].str.lower() == str(day).lower())]
            else:
                subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code)& (df['STATION_ID_SPLITTED']==station_id)& (df[time_column[0]].isin(time_values))]
            subset_df=subset_df.drop_duplicates(subset='id')
            count = subset_df.shape[0]
            ids = subset_df['id'].values
            return count, ids

        pre_early_am_value, pre_early_am_value_ids = get_counts_and_ids(pre_early_am_values)
        early_am_value, early_am_value_ids = get_counts_and_ids(early_am_values)
        am_value, am_value_ids = get_counts_and_ids(am_values)
        midday_value, midday_value_ids = get_counts_and_ids(midday_values)
        pm_value, pm_value_ids = get_counts_and_ids(pm_values)
        evening_value, evening_value_ids = get_counts_and_ids(evening_values)
    #     print(pre_early_am_value,early_am_value,am_value,midday_value,pm_value,evening_value)
        new_df.loc[index, 'CR_Total'] = row['CR_PRE_Early_AM']+row['CR_Early_AM']+row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
        new_df.loc[index, 'CR_AM_Peak'] =row['CR_AM_Peak']
        # new_df.loc[index, 'CR_AM_Peak'] =row['CR_PRE_EARLY_AM']+row['CR_EARLY_AM']+ row['CR_AM_Peak']
        new_df.loc[index, 'DB_PRE_Early_AM_Peak'] = pre_early_am_value
        new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
        new_df.loc[index, 'DB_AM_Peak'] = am_value
        new_df.loc[index, 'DB_Midday'] = midday_value
        new_df.loc[index, 'DB_PM_Peak'] = pm_value
        new_df.loc[index, 'DB_Evening'] = evening_value
        new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value+pre_early_am_value+early_am_value
        unique_station_ids=new_df['STATION_ID_SPLITTED'].unique()

    results = []

    # Iterate over unique station IDs
    for station_id in unique_station_ids:
        # Filter DataFrame for the current station ID
        station_df = new_df[new_df['STATION_ID_SPLITTED'] == station_id]
        
        # Iterate over unique ROUTE_SURVEYEDCode_Splitted for the current station ID
        for route_code in station_df['ROUTE_SURVEYEDCode_Splitted'].unique():
            # Filter rows for the specific route and station
            filtered_df = station_df[station_df['ROUTE_SURVEYEDCode_Splitted'] == route_code]
            
            # Sum numeric columns and convert to a single row
            summed_row = filtered_df.sum(numeric_only=True).to_frame().T
            
            # Add key identifying columns
            summed_row['ROUTE_SURVEYEDCode'] = station_df.iloc[0]['ROUTE_SURVEYEDCode']
            summed_row['STATION_ID'] = station_df.iloc[0]['STATION_ID']
            summed_row['STATION_ID_SPLITTED'] = station_id
            summed_row['ROUTE_SURVEYEDCode_Splitted'] = route_code
            
            # Append the row to results
            results.append(summed_row)

    # Concatenate all results into a new DataFrame
    route_station_wise = pd.concat(results, ignore_index=True)
    route_station_wise.drop(columns=['ROUTE_SURVEYEDCode_Splitted','STATION_ID_SPLITTED'],inplace=True)

    for index, row in route_station_wise.iterrows():
        pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
        early_am_peak_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
        am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
        midday_diff=row['CR_Midday']-row['DB_Midday']    
        pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
        evening_diff=row['CR_Evening']-row['DB_Evening']
        total_diff=row['CR_Total']-row['DB_Total']
#         overall_difference=row['CR_Overall_Goal']-row['DB_Total']
        route_station_wise.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
        route_station_wise.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
        route_station_wise.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
        route_station_wise.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
        route_station_wise.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
        route_station_wise.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
        # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
        route_station_wise.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))

    return route_station_wise

In [30]:
wkday_stationwise_route_df=create_station_wise_route_level_df(wkday_overall_df,weekday_df,None)
wkday_stationwise_route_df

,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_PRE_Early_AM_Peak,DB_Early_AM_Peak,DB_AM_Peak,...,DB_Total,ROUTE_SURVEYEDCode,STATION_ID,PRE_Early_AM_DIFFERENCE,Early_AM_DIFFERENCE,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
0,0.0,0.0,4.0,14.0,28.0,15.0,61.0,0.0,4.0,13.0,...,88.0,UTA_1_703_00,UTA_1_703_19171,0.0,0.0,0.0,0.0,0.0,2.0,2.0
1,0.0,0.0,2.0,10.0,12.0,14.0,38.0,0.0,0.0,1.0,...,45.0,UTA_1_703_00,UTA_1_703_19169,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,0.0,0.0,1.0,15.0,16.0,10.0,42.0,0.0,1.0,4.0,...,47.0,UTA_1_703_00,UTA_1_703_19167,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3,0.0,0.0,2.0,31.0,30.0,18.0,81.0,0.0,0.0,4.0,...,92.0,UTA_1_703_00,UTA_1_703_18384,0.0,0.0,0.0,6.0,2.0,0.0,8.0
4,0.0,0.0,3.0,7.0,5.0,4.0,19.0,0.0,0.0,10.0,...,26.0,UTA_1_703_00,UTA_1_703_18381,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,0.0,0.0,7.0,7.0,7.0,8.0,29.0,2.0,2.0,12.0,...,62.0,UTA_1_750_00,UTA_1_750_21758,0.0,0.0,0.0,0.0,0.0,0.0,0.0
155,0.0,0.0,14.0,8.0,7.0,9.0,38.0,14.0,12.0,14.0,...,94.0,UTA_1_750_00,UTA_1_750_21772,0.0,0.0,0.0,0.0,0.0,0.0,0.0
156,0.0,0.0,13.0,10.0,8.0,9.0,40.0,24.0,16.0,26.0,...,114.0,UTA_1_750_00,UTA_1_750_21757,0.0,0.0,0.0,0.0,0.0,0.0,0.0
157,0.0,0.0,9.0,7.0,7.0,7.0,30.0,34.0,14.0,6.0,...,92.0,UTA_1_750_00,UTA_1_750_21760,0.0,0.0,3.0,0.0,0.0,1.0,4.0


In [22]:
def create_wkend_station_wise_route_level_df(overall_df,df):
    pre_early_am_values = [1]
    early_am_values = [2]
    am_values = [3, 4, 5, 6]
    midday_values = [7, 8, 9, 10, 11]
    pm_values = [12, 13, 14]
    evening_values = [15, 16, 17, 18]

    pre_early_am_column=[0]  #0 is for Pre-Early AM header
    early_am_column=[1]  #1 is for Early AM header
    am_column=[2] #This is for AM header
    midday_column=[3] #this is for MIDDAY header
    pm_column=[4] #this is for PM header
    evening_column=[5] #this is for EVENING header

    def convert_string_to_integer(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return 0

    # Creating new dataframe for specifically AM, PM, MIDDAY, Evenving data and added values from Compeletion Report
    new_df=pd.DataFrame()
    new_df['ROUTE_SURVEYEDCode']=overall_df['LS_NAME_CODE']
    new_df['Day'] = overall_df['DAY']
    new_df['STATION_ID']=overall_df['STATION_ID']
    new_df['STATION_ID_SPLITTED']=overall_df['STATION_ID_SPLITTED']
    new_df['CR_PRE_Early_AM'] = pd.to_numeric(overall_df[pre_early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Early_AM'] = pd.to_numeric(overall_df[early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_AM_Peak'] = pd.to_numeric(overall_df[am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Midday'] = pd.to_numeric(overall_df[midday_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_PM_Peak'] = pd.to_numeric(overall_df[pm_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Evening'] = pd.to_numeric(overall_df[evening_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
    new_df.fillna(0,inplace=True)
    new_df['ROUTE_SURVEYEDCode_Splitted']=new_df['ROUTE_SURVEYEDCode'].apply(edit_ls_code_column)
    
    for index, row in new_df.iterrows():
        route_code = row['ROUTE_SURVEYEDCode_Splitted']
        station_id=row['STATION_ID_SPLITTED']
        day=row['Day']
        def get_counts_and_ids(time_values):
            # Just for SALEM
            # subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code) & (df[time_column[0]].isin(time_values))]
            subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code)& (df['STATION_ID_SPLITTED']==station_id)& (df[time_column[0]].isin(time_values))& 
                         (df['Day'].str.lower() == str(day).lower())]
            subset_df=subset_df.drop_duplicates(subset='id')
            count = subset_df.shape[0]
            ids = subset_df['id'].values
            return count, ids

        pre_early_am_value, pre_early_am_value_ids = get_counts_and_ids(pre_early_am_values)
        early_am_value, early_am_value_ids = get_counts_and_ids(early_am_values)
        am_value, am_value_ids = get_counts_and_ids(am_values)
        midday_value, midday_value_ids = get_counts_and_ids(midday_values)
        pm_value, pm_value_ids = get_counts_and_ids(pm_values)
        evening_value, evening_value_ids = get_counts_and_ids(evening_values)
    #     print(pre_early_am_value,early_am_value,am_value,midday_value,pm_value,evening_value)
        new_df.loc[index, 'CR_Total'] = row['CR_PRE_Early_AM']+row['CR_Early_AM']+row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
        new_df.loc[index, 'CR_AM_Peak'] =row['CR_AM_Peak']
        # new_df.loc[index, 'CR_AM_Peak'] =row['CR_PRE_EARLY_AM']+row['CR_EARLY_AM']+ row['CR_AM_Peak']
        new_df.loc[index, 'DB_PRE_Early_AM_Peak'] = pre_early_am_value
        new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
        new_df.loc[index, 'DB_AM_Peak'] = am_value
        new_df.loc[index, 'DB_Midday'] = midday_value
        new_df.loc[index, 'DB_PM_Peak'] = pm_value
        new_df.loc[index, 'DB_Evening'] = evening_value
        new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value+pre_early_am_value+early_am_value
        unique_station_ids=new_df['STATION_ID_SPLITTED'].unique()

    results = []

    # Iterate over unique station IDs
    for station_id in unique_station_ids:
        # Filter DataFrame for the current station ID
        station_df = new_df[new_df['STATION_ID_SPLITTED'] == station_id]
        
        # Iterate over unique ROUTE_SURVEYEDCode_Splitted for the current station ID
        for route_code in station_df['ROUTE_SURVEYEDCode_Splitted'].unique():
            # Filter rows for the specific route and station
            filtered_df = station_df[station_df['ROUTE_SURVEYEDCode_Splitted'] == route_code]
            
            # Sum numeric columns and convert to a single row
            summed_row = filtered_df.sum(numeric_only=True).to_frame().T
            
            # Add key identifying columns
            summed_row['ROUTE_SURVEYEDCode'] = station_df.iloc[0]['ROUTE_SURVEYEDCode']
            summed_row['STATION_ID'] = station_df.iloc[0]['STATION_ID']
            summed_row['STATION_ID_SPLITTED'] = station_id
            summed_row['ROUTE_SURVEYEDCode_Splitted'] = route_code
            
            # Append the row to results
            results.append(summed_row)

    # Concatenate all results into a new DataFrame
    route_station_wise = pd.concat(results, ignore_index=True)
    route_station_wise.drop(columns=['ROUTE_SURVEYEDCode_Splitted','STATION_ID_SPLITTED'],inplace=True)

    for index, row in route_station_wise.iterrows():
        pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
        early_am_peak_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
        am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
        midday_diff=row['CR_Midday']-row['DB_Midday']    
        pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
        evening_diff=row['CR_Evening']-row['DB_Evening']
        total_diff=row['CR_Total']-row['DB_Total']
#         overall_difference=row['CR_Overall_Goal']-row['DB_Total']
        route_station_wise.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
        route_station_wise.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
        route_station_wise.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
        route_station_wise.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
        route_station_wise.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
        route_station_wise.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
        # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
        route_station_wise.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))

    return route_station_wise

In [33]:
wkend_stationwise_route_df=create_station_wise_route_level_df(wkend_overall_df,weekend_df,'weekend')
# wkend_stationwise_route_df=create_wkend_station_wise_route_level_df(wkend_overall_df,weekend_df)
wkend_stationwise_route_df

,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_PRE_Early_AM_Peak,DB_Early_AM_Peak,DB_AM_Peak,...,DB_Total,ROUTE_SURVEYEDCode,STATION_ID,PRE_Early_AM_DIFFERENCE,Early_AM_DIFFERENCE,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
0,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,0.0,5.0,...,23.0,UTA_1_703_00,UTA_1_703_19171,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,3.0,3.0,3.0,3.0,12.0,1.0,0.0,1.0,...,12.0,UTA_1_703_00,UTA_1_703_19169,0.0,0.0,2.0,0.0,0.0,1.0,3.0
2,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,0.0,0.0,...,8.0,UTA_1_703_00,UTA_1_703_19167,0.0,0.0,3.0,0.0,0.0,2.0,5.0
3,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,0.0,1.0,...,4.0,UTA_1_703_00,UTA_1_703_18384,0.0,0.0,2.0,2.0,2.0,2.0,8.0
4,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,0.0,1.0,...,6.0,UTA_1_703_00,UTA_1_703_18381,0.0,0.0,2.0,0.0,3.0,3.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,0.0,0.0,6.0,6.0,6.0,6.0,24.0,0.0,0.0,4.0,...,10.0,UTA_1_750_00,UTA_1_750_21758,0.0,0.0,2.0,2.0,6.0,4.0,14.0
155,0.0,0.0,6.0,6.0,6.0,6.0,24.0,0.0,0.0,2.0,...,8.0,UTA_1_750_00,UTA_1_750_21772,0.0,0.0,4.0,4.0,6.0,2.0,16.0
156,0.0,0.0,6.0,6.0,6.0,6.0,24.0,0.0,0.0,10.0,...,22.0,UTA_1_750_00,UTA_1_750_21757,0.0,0.0,0.0,0.0,6.0,6.0,12.0
157,0.0,0.0,6.0,6.0,6.0,6.0,24.0,0.0,0.0,12.0,...,18.0,UTA_1_750_00,UTA_1_750_21760,0.0,0.0,0.0,2.0,6.0,4.0,12.0


In [24]:

def create_route_level_df(overall_df,route_df,df):
    pre_early_am_values = [1]
    early_am_values = [2]
    am_values = [3, 4, 5, 6]
    midday_values = [7, 8, 9, 10, 11]
    pm_values = [12, 13, 14]
    evening_values = [15, 16, 17, 18]

    pre_early_am_column=[0]  #0 is for Pre-Early AM header
    early_am_column=[1]  #1 is for Early AM header
    am_column=[2] #This is for AM header
    midday_column=[3] #this is for MIDDAY header
    pm_column=[4] #this is for PM header
    evening_column=[5] #this is for EVENING header

    def convert_string_to_integer(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return 0

    # Creating new dataframe for specifically AM, PM, MIDDAY, Evenving data and added values from Compeletion Report
    new_df=pd.DataFrame()
    new_df['ROUTE_SURVEYEDCode']=overall_df['LS_NAME_CODE']
    new_df['CR_PRE_Early_AM'] = pd.to_numeric(overall_df[pre_early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Early_AM'] = pd.to_numeric(overall_df[early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_AM_Peak'] = pd.to_numeric(overall_df[am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Midday'] = pd.to_numeric(overall_df[midday_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_PM_Peak'] = pd.to_numeric(overall_df[pm_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Evening'] = pd.to_numeric(overall_df[evening_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
    new_df.fillna(0,inplace=True)
    #  new code added for merging the same ROUTE_SURVEYEDCode 
    new_df = new_df.groupby('ROUTE_SURVEYEDCode', as_index=False).sum()
    new_df.reset_index(drop=True, inplace=True)

    # adding values for AM, PM, MIDDAY and Evening from Database file to new Dataframe
    for index, row in new_df.iterrows():
        route_code = row['ROUTE_SURVEYEDCode']

        def get_counts_and_ids(time_values):
            # Just for SALEM
            # subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code) & (df[time_column[0]].isin(time_values))]
            subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code) & (df[time_column[0]].isin(time_values))]
            subset_df=subset_df.drop_duplicates(subset='id')
            count = subset_df.shape[0]
            ids = subset_df['id'].values
            return count, ids

        pre_early_am_value, pre_early_am_value_ids = get_counts_and_ids(pre_early_am_values)
        early_am_value, early_am_value_ids = get_counts_and_ids(early_am_values)
        am_value, am_value_ids = get_counts_and_ids(am_values)
        midday_value, midday_value_ids = get_counts_and_ids(midday_values)
        pm_value, pm_value_ids = get_counts_and_ids(pm_values)
        evening_value, evening_value_ids = get_counts_and_ids(evening_values)
    #     print(pre_early_am_value,early_am_value,am_value,midday_value,pm_value,evening_value)
        new_df.loc[index, 'CR_Total'] = row['CR_PRE_Early_AM']+row['CR_Early_AM']+row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
        new_df.loc[index, 'CR_AM_Peak'] =row['CR_AM_Peak']
        # new_df.loc[index, 'CR_AM_Peak'] =row['CR_PRE_EARLY_AM']+row['CR_EARLY_AM']+ row['CR_AM_Peak']
        new_df.loc[index, 'DB_PRE_Early_AM_Peak'] = pre_early_am_value
        new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
        new_df.loc[index, 'DB_AM_Peak'] = am_value
        new_df.loc[index, 'DB_Midday'] = midday_value
        new_df.loc[index, 'DB_PM_Peak'] = pm_value
        new_df.loc[index, 'DB_Evening'] = evening_value
        new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value+pre_early_am_value+early_am_value

        
        # # Join the IDs as a comma-separated string
        new_df.loc[index, 'DB_PRE_Early_AM_IDS'] = ', '.join(map(str, pre_early_am_value_ids))
        new_df.loc[index, 'DB_Early_AM_IDS'] = ', '.join(map(str, early_am_value_ids))
        new_df.loc[index, 'DB_AM_IDS'] = ', '.join(map(str, am_value_ids))
        new_df.loc[index, 'DB_Midday_IDS'] = ', '.join(map(str, midday_value_ids))
        new_df.loc[index, 'DB_PM_IDS'] = ', '.join(map(str, pm_value_ids))
        new_df.loc[index, 'DB_Evening_IDS'] = ', '.join(map(str, evening_value_ids))

    # new_df.to_csv('Time Base Comparison(Over All).csv',index=False)

    # Route Level Comparison
    # Just for SALEM because in SALEM Code values are already splitted
    # new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode']
    new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(x.split('_')[:-1]) )

    # creating new dataframe for ROUTE_LEVEL_Comparison
    route_level_df=pd.DataFrame()

    unique_routes=new_df['ROUTE_SURVEYEDCode_Splited'].unique()

    route_level_df['ROUTE_SURVEYEDCode']=unique_routes

    # Have to change the name accordingly
    route_df.rename(columns={'ROUTE_TOTAL':'CR_Overall_Goal','ETC_ROUTE_ID':'ROUTE_SURVEYEDCode'},inplace=True)

    route_df.dropna(subset=['ROUTE_SURVEYEDCode'],inplace=True)
    route_level_df=pd.merge(route_level_df,route_df[['ROUTE_SURVEYEDCode','CR_Overall_Goal']],on='ROUTE_SURVEYEDCode')

    route_level_df=route_level_df.groupby('ROUTE_SURVEYEDCode', as_index=False).sum()
    route_level_df.reset_index(drop=True, inplace=True)

    # adding values from database file and compeletion report for Route_Level
    for index , row in route_level_df.iterrows():
        subset_df=new_df[new_df['ROUTE_SURVEYEDCode_Splited']==row['ROUTE_SURVEYEDCode']]
        # sum_per_route_cr = subset_df[['CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total','Overall Goal']].sum()



        sum_per_route_cr = subset_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
        # sum_per_route_cr = subset_df[['CR_EARLY_AM','CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
        sum_per_route_db = subset_df[['DB_PRE_Early_AM_Peak','DB_Early_AM_Peak','DB_AM_Peak', 'DB_Midday', 'DB_PM_Peak', 'DB_Evening','DB_Total']].sum()

        route_level_df.loc[index,'CR_PRE_Early_AM']=sum_per_route_cr['CR_PRE_Early_AM']
        route_level_df.loc[index,'CR_Early_AM']=sum_per_route_cr['CR_Early_AM']
        route_level_df.loc[index,'CR_AM_Peak']=sum_per_route_cr['CR_AM_Peak']
        route_level_df.loc[index,'CR_Midday']=sum_per_route_cr['CR_Midday']
        route_level_df.loc[index,'CR_PM_Peak']=sum_per_route_cr['CR_PM_Peak']
        route_level_df.loc[index,'CR_Evening']=sum_per_route_cr['CR_Evening']
        route_level_df.loc[index,'CR_Total']=sum_per_route_cr['CR_Total']
        
        route_level_df.loc[index,'DB_PRE_Early_AM_Peak']=sum_per_route_db['DB_PRE_Early_AM_Peak']
        route_level_df.loc[index,'DB_Early_AM_Peak']=sum_per_route_db['DB_Early_AM_Peak']
        route_level_df.loc[index,'DB_AM_Peak']=sum_per_route_db['DB_AM_Peak']
        route_level_df.loc[index,'DB_Midday']=sum_per_route_db['DB_Midday']
        route_level_df.loc[index,'DB_PM_Peak']=sum_per_route_db['DB_PM_Peak']
        route_level_df.loc[index,'DB_Evening']=sum_per_route_db['DB_Evening']
        route_level_df.loc[index,'DB_Total']=sum_per_route_db['DB_Total']   
        route_level_df.loc[index,'DB_PRE_Early_AM_IDS']=', '.join(str(value) for value in subset_df['DB_PRE_Early_AM_IDS'].values)    
        route_level_df.loc[index,'DB_Early_AM_IDS']=', '.join(str(value) for value in subset_df['DB_Early_AM_IDS'].values)    
        route_level_df.loc[index,'DB_AM_IDS']=', '.join(str(value) for value in subset_df['DB_AM_IDS'].values)    
        route_level_df.loc[index,'DB_Midday_IDS']=', '.join(str(value) for value in subset_df['DB_Midday_IDS'].values)    
        route_level_df.loc[index,'DB_PM_IDS']=', '.join(str(value) for value in subset_df['DB_PM_IDS'].values)    
        route_level_df.loc[index,'DB_Evening_IDS']=', '.join(str(value) for value in subset_df['DB_Evening_IDS'].values)

    # route_level_df.to_csv('Route Level Comparison(Value_Check).csv',index=False)
        
    # calculating the difference between values of database and compeletion report for Route_Level
    for index, row in route_level_df.iterrows():
        pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
        early_am_peak_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
        am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
        midday_diff=row['CR_Midday']-row['DB_Midday']    
        pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
        evening_diff=row['CR_Evening']-row['DB_Evening']
        total_diff=row['CR_Total']-row['DB_Total']
        overall_difference=row['CR_Overall_Goal']-row['DB_Total']
        route_level_df.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
        route_level_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
        route_level_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
        route_level_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
        route_level_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
        route_level_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
        # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
        route_level_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))
        route_level_df.loc[index, 'Overall_Goal_DIFFERENCE'] = math.ceil(max(0,overall_difference))
    return route_level_df



In [25]:
wkday_route_level =create_route_level_df(wkday_overall_df,wkday_route_df,weekday_df)
wkday_route_level

,ROUTE_SURVEYEDCode,CR_Overall_Goal,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_PRE_Early_AM_Peak,...,DB_PM_IDS,DB_Evening_IDS,PRE_Early_AM_DIFFERENCE,Early_AM_DIFFERENCE,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE,Overall_Goal_DIFFERENCE
0,UTA_1_701,892,0.0,0.0,127.0,225.0,169.0,160.0,681.0,22.0,...,"6680, 6699, 6709, 6718, 6737, 6740, 6759, 6761...","6870, 6872, 6877, 6881, 6898, 6909, 6915, 6923...",0.0,0.0,0.0,0.0,0.0,20.0,20.0,135.0
1,UTA_1_703,1368,0.0,0.0,191.0,331.0,259.0,224.0,1005.0,21.0,...,"11708, 11714, 11748, 14328, 16653, 16671, 1667...","6910, 11878, 16895, 16902, 19335, 19412, 19420...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,108.0
2,UTA_1_704,1081,0.0,0.0,133.0,273.0,196.0,184.0,786.0,26.0,...,"11608, 20654, 20669, 21821, 21877, 21896, 2190...","20777, 21917, 21924, 21925, 21927, 21928, 2192...",0.0,0.0,0.0,27.0,0.0,43.0,70.0,173.0
3,UTA_1_750,1369,0.0,0.0,239.0,242.0,277.0,214.0,972.0,91.0,...,"6981, 6993, 7534, 7548, 7553, 7555, 7560, 7562...","6874, 6890, 6900, 7710, 7741, 7745, 7750, 7762...",0.0,0.0,52.0,0.0,0.0,0.0,52.0,158.0


In [27]:

def create_wkend_route_level_df(overall_df,route_df,df):
    pre_early_am_values = [1]
    early_am_values = [2]
    am_values = [3, 4, 5, 6]
    midday_values = [7, 8, 9, 10, 11]
    pm_values = [12, 13, 14]
    evening_values = [15, 16, 17, 18]

    pre_early_am_column=[0]  #0 is for Pre-Early AM header
    early_am_column=[1]  #1 is for Early AM header
    am_column=[2] #This is for AM header
    midday_column=[3] #this is for MIDDAY header
    pm_column=[4] #this is for PM header
    evening_column=[5] #this is for EVENING header

    def convert_string_to_integer(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return 0

    # Creating new dataframe for specifically AM, PM, MIDDAY, Evenving data and added values from Compeletion Report
    new_df=pd.DataFrame()
    new_df['ROUTE_SURVEYEDCode']=overall_df['LS_NAME_CODE']
    new_df['Day']=overall_df['DAY']
    new_df['CR_PRE_Early_AM'] = pd.to_numeric(overall_df[pre_early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Early_AM'] = pd.to_numeric(overall_df[early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_AM_Peak'] = pd.to_numeric(overall_df[am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Midday'] = pd.to_numeric(overall_df[midday_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_PM_Peak'] = pd.to_numeric(overall_df[pm_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Evening'] = pd.to_numeric(overall_df[evening_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
    new_df.fillna(0,inplace=True)
    #  new code added for merging the same ROUTE_SURVEYEDCode 
    new_df = new_df.groupby('ROUTE_SURVEYEDCode', as_index=False).sum()
    new_df.reset_index(drop=True, inplace=True)

    # adding values for AM, PM, MIDDAY and Evening from Database file to new Dataframe
    for index, row in new_df.iterrows():
        route_code = row['ROUTE_SURVEYEDCode']
        day = row['Day']

        def get_counts_and_ids(time_values):
            # Just for SALEM
            # subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code) & (df[time_column[0]].isin(time_values))]
            subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code) & (df[time_column[0]].isin(time_values))&(df['Day'].str.lower() == str(day).lower())]
            subset_df=subset_df.drop_duplicates(subset='id')
            count = subset_df.shape[0]
            ids = subset_df['id'].values
            return count, ids

        pre_early_am_value, pre_early_am_value_ids = get_counts_and_ids(pre_early_am_values)
        early_am_value, early_am_value_ids = get_counts_and_ids(early_am_values)
        am_value, am_value_ids = get_counts_and_ids(am_values)
        midday_value, midday_value_ids = get_counts_and_ids(midday_values)
        pm_value, pm_value_ids = get_counts_and_ids(pm_values)
        evening_value, evening_value_ids = get_counts_and_ids(evening_values)
    #     print(pre_early_am_value,early_am_value,am_value,midday_value,pm_value,evening_value)
        new_df.loc[index, 'CR_Total'] = row['CR_PRE_Early_AM']+row['CR_Early_AM']+row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
        new_df.loc[index, 'CR_AM_Peak'] =row['CR_AM_Peak']
        # new_df.loc[index, 'CR_AM_Peak'] =row['CR_PRE_EARLY_AM']+row['CR_EARLY_AM']+ row['CR_AM_Peak']
        new_df.loc[index, 'DB_PRE_Early_AM_Peak'] = pre_early_am_value
        new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
        new_df.loc[index, 'DB_AM_Peak'] = am_value
        new_df.loc[index, 'DB_Midday'] = midday_value
        new_df.loc[index, 'DB_PM_Peak'] = pm_value
        new_df.loc[index, 'DB_Evening'] = evening_value
        new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value+pre_early_am_value+early_am_value

        
        # # Join the IDs as a comma-separated string
        new_df.loc[index, 'DB_PRE_Early_AM_IDS'] = ', '.join(map(str, pre_early_am_value_ids))
        new_df.loc[index, 'DB_Early_AM_IDS'] = ', '.join(map(str, early_am_value_ids))
        new_df.loc[index, 'DB_AM_IDS'] = ', '.join(map(str, am_value_ids))
        new_df.loc[index, 'DB_Midday_IDS'] = ', '.join(map(str, midday_value_ids))
        new_df.loc[index, 'DB_PM_IDS'] = ', '.join(map(str, pm_value_ids))
        new_df.loc[index, 'DB_Evening_IDS'] = ', '.join(map(str, evening_value_ids))

    # new_df.to_csv('Time Base Comparison(Over All).csv',index=False)

    # Route Level Comparison
    # Just for SALEM because in SALEM Code values are already splitted
    # new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode']
    new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(x.split('_')[:-1]) )

    # creating new dataframe for ROUTE_LEVEL_Comparison
    route_level_df=pd.DataFrame()

    unique_routes=new_df['ROUTE_SURVEYEDCode_Splited'].unique()

    route_level_df['ROUTE_SURVEYEDCode']=unique_routes

    # Have to change the name accordingly
    route_df.rename(columns={'ROUTE_TOTAL':'CR_Overall_Goal','ETC_ROUTE_ID':'ROUTE_SURVEYEDCode'},inplace=True)

    route_df.dropna(subset=['ROUTE_SURVEYEDCode'],inplace=True)
    route_level_df=pd.merge(route_level_df,route_df[['ROUTE_SURVEYEDCode','CR_Overall_Goal']],on='ROUTE_SURVEYEDCode')

    route_level_df=route_level_df.groupby('ROUTE_SURVEYEDCode', as_index=False).sum()
    route_level_df.reset_index(drop=True, inplace=True)

    # adding values from database file and compeletion report for Route_Level
    for index , row in route_level_df.iterrows():
        subset_df=new_df[new_df['ROUTE_SURVEYEDCode_Splited']==row['ROUTE_SURVEYEDCode']]
        # sum_per_route_cr = subset_df[['CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total','Overall Goal']].sum()



        sum_per_route_cr = subset_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
        # sum_per_route_cr = subset_df[['CR_EARLY_AM','CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
        sum_per_route_db = subset_df[['DB_PRE_Early_AM_Peak','DB_Early_AM_Peak','DB_AM_Peak', 'DB_Midday', 'DB_PM_Peak', 'DB_Evening','DB_Total']].sum()

        route_level_df.loc[index,'CR_PRE_Early_AM']=sum_per_route_cr['CR_PRE_Early_AM']
        route_level_df.loc[index,'CR_Early_AM']=sum_per_route_cr['CR_Early_AM']
        route_level_df.loc[index,'CR_AM_Peak']=sum_per_route_cr['CR_AM_Peak']
        route_level_df.loc[index,'CR_Midday']=sum_per_route_cr['CR_Midday']
        route_level_df.loc[index,'CR_PM_Peak']=sum_per_route_cr['CR_PM_Peak']
        route_level_df.loc[index,'CR_Evening']=sum_per_route_cr['CR_Evening']
        route_level_df.loc[index,'CR_Total']=sum_per_route_cr['CR_Total']
        
        route_level_df.loc[index,'DB_PRE_Early_AM_Peak']=sum_per_route_db['DB_PRE_Early_AM_Peak']
        route_level_df.loc[index,'DB_Early_AM_Peak']=sum_per_route_db['DB_Early_AM_Peak']
        route_level_df.loc[index,'DB_AM_Peak']=sum_per_route_db['DB_AM_Peak']
        route_level_df.loc[index,'DB_Midday']=sum_per_route_db['DB_Midday']
        route_level_df.loc[index,'DB_PM_Peak']=sum_per_route_db['DB_PM_Peak']
        route_level_df.loc[index,'DB_Evening']=sum_per_route_db['DB_Evening']
        route_level_df.loc[index,'DB_Total']=sum_per_route_db['DB_Total']   
        route_level_df.loc[index,'DB_PRE_Early_AM_IDS']=', '.join(str(value) for value in subset_df['DB_PRE_Early_AM_IDS'].values)    
        route_level_df.loc[index,'DB_Early_AM_IDS']=', '.join(str(value) for value in subset_df['DB_Early_AM_IDS'].values)    
        route_level_df.loc[index,'DB_AM_IDS']=', '.join(str(value) for value in subset_df['DB_AM_IDS'].values)    
        route_level_df.loc[index,'DB_Midday_IDS']=', '.join(str(value) for value in subset_df['DB_Midday_IDS'].values)    
        route_level_df.loc[index,'DB_PM_IDS']=', '.join(str(value) for value in subset_df['DB_PM_IDS'].values)    
        route_level_df.loc[index,'DB_Evening_IDS']=', '.join(str(value) for value in subset_df['DB_Evening_IDS'].values)

    # route_level_df.to_csv('Route Level Comparison(Value_Check).csv',index=False)
        
    # calculating the difference between values of database and compeletion report for Route_Level
    for index, row in route_level_df.iterrows():
        pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
        early_am_peak_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
        am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
        midday_diff=row['CR_Midday']-row['DB_Midday']    
        pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
        evening_diff=row['CR_Evening']-row['DB_Evening']
        total_diff=row['CR_Total']-row['DB_Total']
        overall_difference=row['CR_Overall_Goal']-row['DB_Total']
        route_level_df.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
        route_level_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
        route_level_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
        route_level_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
        route_level_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
        route_level_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
        # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
        route_level_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))
        route_level_df.loc[index, 'Overall_Goal_DIFFERENCE'] = math.ceil(max(0,overall_difference))
    return route_level_df



In [28]:
wkend_route_level =create_wkend_route_level_df(wkend_overall_df,wkend_route_df,weekend_df)
wkend_route_level

,ROUTE_SURVEYEDCode,CR_Overall_Goal,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_PRE_Early_AM_Peak,...,DB_PM_IDS,DB_Evening_IDS,PRE_Early_AM_DIFFERENCE,Early_AM_DIFFERENCE,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE,Overall_Goal_DIFFERENCE
0,UTA_1_701,1500,0.0,0.0,150.0,150.0,150.0,150.0,600.0,0.0,...,",",",",0.0,0.0,150.0,150.0,150.0,150.0,600.0,1500.0
1,UTA_1_703,1590,0.0,0.0,159.0,159.0,159.0,159.0,636.0,0.0,...,",",",",0.0,0.0,159.0,159.0,159.0,159.0,636.0,1590.0
2,UTA_1_704,1170,0.0,0.0,117.0,117.0,117.0,117.0,468.0,0.0,...,",",",",0.0,0.0,117.0,117.0,117.0,117.0,468.0,1170.0
3,UTA_1_750,1020,0.0,0.0,102.0,102.0,102.0,102.0,408.0,0.0,...,",",",",0.0,0.0,102.0,102.0,102.0,102.0,408.0,1020.0
